In [10]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

In [11]:
bin_path = '../data/movie_embeddings.bin'

with open(bin_path, 'rb') as f:
    shape = np.fromfile(f, dtype=np.uint32, count=2)
    num_movies, vector_dim = shape[0], shape[1]
    
    data = np.fromfile(f, dtype=np.float32)
    
    embeddings_matrix = data.reshape(num_movies, vector_dim)

In [12]:
kmeans = KMeans(n_clusters=15, init='k-means++', random_state=0)
kmeans.fit(embeddings_matrix)

KMeans(n_clusters=15, random_state=0)

In [ ]:
distances = pairwise_distances(kmeans.cluster_centers_, embeddings_matrix, metric='cosine')

closest_indices = np.argmin(distances, axis=1)

df_mapping = pd.read_csv('../data/movie_mapping.csv')

starter_movies = df_mapping.iloc[closest_indices].copy()

display(starter_movies)
starter_ids = starter_movies['id'].tolist()
print("\nСписок ID для обновления базы данных:")
print(starter_ids)

,id,title
13295,11297,Спасти зеленую планету!
12142,13492,Граница
16306,18357,Деревенские медведи
3582,10998,Роковое влечение
9224,776835,Как слониха упала с неба
9209,150686,Паршивые овцы
12125,32261,Скандал
12859,1037348,Мой счастливый брак
15868,105001,Ледяная комета
15627,8088,Разомкнутые объятия



Список ID для обновления базы данных:
[11297, 13492, 18357, 10998, 776835, 150686, 32261, 1037348, 105001, 8088, 1010928, 602269, 50725, 615658, 1220410]


In [14]:
TOP_K_NEIGHBORS = 30 

sorted_indices = np.argsort(distances, axis=1)

starter_ids = []
starter_movies_list = []

df = pd.read_csv('../data/tmdb_movies_ru.csv')

for i in range(15):
    cluster_top_k_indices = sorted_indices[i, :TOP_K_NEIGHBORS]

    candidates = df.iloc[cluster_top_k_indices]

    best_candidate = candidates.sort_values(by='popularity', ascending=False).iloc[0]
    
    starter_ids.append(best_candidate['id'])
    starter_movies_list.append(best_candidate)

final_starters_df = pd.DataFrame(starter_movies_list)[['id', 'title', 'popularity']]
display(final_starters_df)

,id,title,popularity
132,438631,Дюна,34.1214
59,155,Тёмный рыцарь,57.5176
99,713704,Восстание зловещих мертвецов,39.7248
554,1327862,Сожалею о тебе,15.3473
44,1007757,В чужой шкуре,71.2324
318,1495,Царство небесное,20.8734
266,616037,Тор: Любовь и гром,23.1575
76,329505,Грех Лолы,46.4964
925,79,Герой,11.5533
55,1595852,Бульвар,62.0562
